# Binoculars on Google Colab

This notebook installs and runs the original Binoculars repository with its default Falcon-7B + Falcon-7B-Instruct model pair.

For the closest match to the published setup:
- In Colab, choose `Runtime` -> `Change runtime type` -> `GPU` before running.
- If you hit an out-of-memory error, reconnect to a larger GPU runtime and rerun from the top.
- The first model load can take a while because the Falcon weights are large.

Note: current Colab runtimes use Python 3.12, and the repo's old `transformers` pin can fail there while building `tokenizers`. This notebook installs a current compatible `transformers` stack first, then installs Binoculars itself without reusing the old dependency pin.

In [1]:
import os

REPO_URL = "https://github.com/ahans30/Binoculars.git"
REPO_DIR = "/content/Binoculars"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/Binoculars

!pip -q install -U pip
!pip -q install -U setuptools wheel
!pip -q install "jedi>=0.19" "transformers>=4.46,<5" "tokenizers>=0.20" accelerate sentencepiece datasets numpy gradio gradio_client scikit-learn seaborn pandas huggingface_hub[hf_xet]
!pip -q install -e . --no-deps


Cloning into '/content/Binoculars'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 208 (delta 72), reused 64 (delta 64), pack-reused 119 (from 1)
Receiving objects: 100% (208/208), 54.08 MiB | 38.56 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/Binoculars
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 53.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for Binoculars (pyproject.toml) ... done


In [2]:
import os
import platform
import shutil
import subprocess
import torch

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "This Colab runtime is CPU-only. Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU, then rerun from the top."
    )

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)


Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA device count: 1
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Evaluate HC3 CSVs

Upload `evaluate_samples.py` plus one or more HC3 CSVs created by `create_hc3_sample.py`, such as `hc3_unified_1000_seed42.csv` or any `hc3_<source>_200_seed42.csv`.

Each CSV must use the HC3 wide schema: `hc3_row_id`, `source`, `question`, `human_answers`, and `chatgpt_answers`. The cell expands those answer-list columns into row-wise `text` and `label` samples before running Binoculars.

In [3]:
from google.colab import files
from pathlib import Path
import ast
import json
import os
import re
import pandas as pd
import subprocess
import sys

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

script_uploaded_name = None
csv_uploaded_names = []

for filename in uploaded.keys():
    if filename.endswith(".py"):
        script_uploaded_name = filename
    elif filename.endswith(".csv"):
        csv_uploaded_names.append(filename)

if not script_uploaded_name:
    raise FileNotFoundError("Upload evaluate_samples.py along with the HC3 CSV file(s).")
if not csv_uploaded_names:
    raise FileNotFoundError("Upload at least one HC3 CSV file created by create_hc3_sample.py.")

script_destination_path = os.path.join(REPO_DIR, "evaluate_samples.py")
with open(script_destination_path, "wb") as f:
    f.write(uploaded[script_uploaded_name])
print(f"Saved '{script_uploaded_name}' as 'evaluate_samples.py' to '{script_destination_path}'")

os.chdir(REPO_DIR)

raw_input_dir = Path(REPO_DIR) / "hc3_uploaded_inputs"
raw_input_dir.mkdir(exist_ok=True)

def safe_stem(filename):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(filename).stem)

def parse_answer_list(value):
    if isinstance(value, list):
        parsed = value
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed = [value]
    elif pd.isna(value):
        return []
    else:
        parsed = value

    if isinstance(parsed, list):
        return [str(item).strip() for item in parsed if pd.notna(item) and str(item).strip()]
    if pd.notna(parsed) and str(parsed).strip():
        return [str(parsed).strip()]
    return []

def build_hc3_eval_dataframe(df):
    required_columns = {"hc3_row_id", "source", "question", "human_answers", "chatgpt_answers"}
    missing_columns = sorted(required_columns.difference(df.columns))
    if missing_columns:
        raise ValueError(
            "Unsupported CSV format. Expected an HC3 CSV created by create_hc3_sample.py. "
            f"Missing columns: {missing_columns}"
        )

    rows = []
    for row in df.itertuples(index=False):
        base = {
            "hc3_id": row.hc3_row_id,
            "hc3_row_id": row.hc3_row_id,
            "source": row.source,
            "question": row.question,
        }
        for column_name, label in [("human_answers", "human"), ("chatgpt_answers", "ai")]:
            for answer_index, text in enumerate(parse_answer_list(getattr(row, column_name))):
                rows.append({
                    **base,
                    "answer_type": label,
                    "answer_index": answer_index,
                    "text": text,
                    "label": label,
                })

    eval_df = pd.DataFrame(rows)
    if eval_df.empty:
        raise ValueError("The uploaded HC3 CSV did not contain any answer text to score.")
    return eval_df

def merge_scores(eval_df, scored_df):
    if "sample_id" in scored_df.columns:
        return pd.merge(eval_df, scored_df, on="sample_id", how="left", validate="one_to_one", suffixes=("", "_scored"))
    if len(scored_df) == len(eval_df):
        scored_trimmed = scored_df.drop(columns=[col for col in ["text", "label"] if col in scored_df.columns])
        return pd.concat([eval_df.reset_index(drop=True), scored_trimmed.reset_index(drop=True)], axis=1)

    eval_with_index = eval_df.copy()
    scored_with_index = scored_df.copy()
    eval_with_index["_merge_idx"] = eval_with_index.groupby(["text", "label"]).cumcount()
    scored_with_index["_merge_idx"] = scored_with_index.groupby(["text", "label"]).cumcount()
    return pd.merge(
        eval_with_index,
        scored_with_index,
        on=["text", "label", "_merge_idx"],
        how="left",
        validate="one_to_one",
        suffixes=("", "_scored"),
    ).drop(columns=["_merge_idx"])

all_results = {}

for csv_uploaded_name in csv_uploaded_names:
    run_name = safe_stem(csv_uploaded_name)
    print()
    print(f"Evaluating {csv_uploaded_name}")

    raw_input_path = raw_input_dir / f"{run_name}.csv"
    with open(raw_input_path, "wb") as f:
        f.write(uploaded[csv_uploaded_name])

    original_df = pd.read_csv(raw_input_path)
    eval_df = build_hc3_eval_dataframe(original_df).reset_index(drop=True)
    eval_df["sample_id"] = range(len(eval_df))

    data_input_path = Path(REPO_DIR) / f"{run_name}_binoculars_input.csv"
    output_dir = f"{run_name}_eval"
    eval_df.to_csv(data_input_path, index=False)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Prepared {len(eval_df)} evaluation rows in '{data_input_path}'")

    cmd = [
        sys.executable, "evaluate_samples.py",
        "--input", str(data_input_path),
        "--text-column", "text",
        "--label-column", "label",
        "--mode", "low-fpr",
        "--output-dir", output_dir,
    ]
    subprocess.run(cmd, check=True)

    with open(Path(output_dir) / "summary.json", "r") as f:
        summary = json.load(f)

    print("Summary:")
    print(json.dumps(summary, indent=2))

    scored_df = pd.read_csv(Path(output_dir) / "scored_samples.csv")
    result_df = merge_scores(eval_df, scored_df)
    all_results[run_name] = result_df

    summary_columns = [
        column for column in [
            "hc3_id", "hc3_row_id", "source", "question", "answer_type", "answer_index", "label", "score", "prediction_text"
        ] if column in result_df.columns
    ]
    display(result_df[summary_columns])

    label_num = pd.to_numeric(result_df["label"].replace({"human": 0, "ai": 1}), errors="coerce")
    prediction_num = pd.to_numeric(result_df["prediction"], errors="coerce")
    mistakes = result_df[label_num.ne(prediction_num)]

    print("Mistakes:")
    mistake_columns = [
        column for column in [
            "hc3_id", "hc3_row_id", "source", "question", "answer_type", "answer_index", "label", "score", "prediction_text", "text"
        ] if column in mistakes.columns
    ]
    display(mistakes[mistake_columns])

print()
print("Finished evaluating:", list(all_results.keys()))


Saving hc3_unified_t5_perturbed_ai_clean.csv to hc3_unified_t5_perturbed_ai_clean.csv
Saving evaluate_samples.py to evaluate_samples.py
Uploaded: ['hc3_unified_t5_perturbed_ai_clean.csv', 'evaluate_samples.py']
Saved 'evaluate_samples.py' as 'evaluate_samples.py' to '/content/Binoculars/evaluate_samples.py'

Evaluating hc3_unified_t5_perturbed_ai_clean.csv
Prepared 33097 evaluation rows in '/content/Binoculars/hc3_unified_t5_perturbed_ai_clean_binoculars_input.csv'
Summary:
{
  "experiment_name": "hc3_unified_t5_perturbed_ai_clean_binoculars_input.csv",
  "detection_method": "Binoculars",
  "model_used": "",
  "dataset_used": "hc3_unified_t5_perturbed_ai_clean_binoculars_input",
  "num_samples": 33097,
  "additional_details": {
    "additional_models_used": [],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.7619206402134044,
    "accuracy": 0.8705622866120797,
    "precision": 0.968767665347654,
    "recall": 0.6278622458325701,
    "auc_roc": 0.9748988399032564
  },
  "me

,hc3_id,hc3_row_id,source,question,answer_type,answer_index,label,score,prediction_text
0,19141,19141,finance,Historical P/E ratios of small-cap vs. large-c...,human,0,human,0.925532,Most likely human-written
1,19141,19141,finance,Historical P/E ratios of small-cap vs. large-c...,ai,0,ai,0.677725,Most likely AI-generated
2,19142,19142,finance,Should you co-sign a personal loan for a frien...,human,0,human,1.011905,Most likely human-written
3,19142,19142,finance,Should you co-sign a personal loan for a frien...,ai,0,ai,0.751497,Most likely AI-generated
4,19147,19147,finance,"For a car loan, how much should I get preappro...",human,0,human,0.987342,Most likely human-written
...,...,...,...,...,...,...,...,...,...
33092,19136,19136,wiki_csai,"Please explain what is ""Punched cards""",ai,0,ai,0.743976,Most likely AI-generated
33093,19138,19138,wiki_csai,"Please explain what is ""BBC Model B""",human,0,human,0.956522,Most likely human-written
33094,19138,19138,wiki_csai,"Please explain what is ""BBC Model B""",ai,0,ai,0.919492,Most likely human-written
33095,19139,19139,wiki_csai,"Please explain what is ""O level""",human,0,human,1.046053,Most likely human-written


Mistakes:


/tmp/ipykernel_861/521460367.py:166: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  label_num = pd.to_numeric(result_df["label"].replace({"human": 0, "ai": 1}), errors="coerce")


,hc3_id,hc3_row_id,source,question,answer_type,answer_index,label,score,prediction_text,text
16,19154,19154,finance,incorrect printed information on check stock,ai,1,ai,0.910314,Most likely human-written,If you have check stock with incorrect informa...
28,19165,19165,finance,Where is my dividend?,ai,1,ai,0.944162,Most likely human-written,"I'm not sure, but I need more information to u..."
121,19279,19279,finance,Paid part of my state refund back last year; n...,human,0,human,0.822222,Most likely AI-generated,http://www.irs.gov/taxtopics/tc503.html says y...
148,19311,19311,finance,Is it beneficial to convert non-investment rea...,ai,1,ai,0.874346,Most likely human-written,Converting non-investment real estate to renta...
153,19313,19313,finance,How to improve credit score and borrow money,ai,1,ai,0.864583,Most likely human-written,Improving your credit score is important becau...
...,...,...,...,...,...,...,...,...,...,...
33021,19074,19074,wiki_csai,"Please explain what is ""Rózsa Péter""",human,0,human,0.826531,Most likely AI-generated,"Rózsa Péter, born Rózsa Politzer, (17 February..."
33041,19096,19096,wiki_csai,"Please explain what is ""Programming language""",human,0,human,0.834320,Most likely AI-generated,A programming language is a system of notation...
33069,19118,19118,wiki_csai,"Please explain what is ""Applied mathematics""",human,0,human,0.835294,Most likely AI-generated,Applied mathematics is the application of math...
33077,19124,19124,wiki_csai,"Please explain what is ""Medical image computing""",human,0,human,0.839744,Most likely AI-generated,Medical image computing (MIC) is an interdisci...



Finished evaluating: ['hc3_unified_t5_perturbed_ai_clean']


In [4]:
from pathlib import Path
import json

EXPERIMENT_NAME = "Binoculars HC3 evaluation 2"
DETECTION_METHOD = "Binoculars"
MODEL_USED = "tiiuae/falcon-7b + tiiuae/falcon-7b-instruct"
ADDITIONAL_MODELS_USED = ["tiiuae/falcon-7b", "tiiuae/falcon-7b-instruct"]
NOTES = "Zero-shot Binoculars detector using the configured threshold mode."

METRIC_KEYS = ("f1", "accuracy", "precision", "recall", "auc_roc")

def metric_block(summary, key):
    raw = summary.get(key)
    if not isinstance(raw, dict):
        raw = {}
    out = {}
    for metric_key in METRIC_KEYS:
        value = raw.get(metric_key)
        out[metric_key] = float(value) if value is not None else None
    return out

def build_experiment_report(run_name, summary):
    return {
        "experiment_name": EXPERIMENT_NAME,
        "detection_method": DETECTION_METHOD,
        "model_used": MODEL_USED,
        "dataset_used": run_name,
        "num_samples": int(summary.get("num_samples") or summary.get("count") or 0),
        "additional_details": {
            "additional_models_used": ADDITIONAL_MODELS_USED,
            "notes": NOTES,
        },
        "metrics_at_1pct_fpr": metric_block(summary, "metrics_at_1pct_fpr"),
        "metrics_at_0.1pct_fpr": metric_block(summary, "metrics_at_0.1pct_fpr"),
    }

if "csv_uploaded_names" in globals():
    run_names = [safe_stem(filename) for filename in csv_uploaded_names]
else:
    run_names = [path.name.removesuffix("_eval") for path in Path(REPO_DIR).glob("*_eval")]

reports = []
for run_name in run_names:
    summary_path = Path(REPO_DIR) / f"{run_name}_eval" / "summary.json"
    if not summary_path.exists():
        print(f"Skipping {run_name}: missing {summary_path}")
        continue

    with open(summary_path, "r") as f:
        summary = json.load(f)

    report = build_experiment_report(run_name, summary)
    reports.append(report)

    report_path = Path(REPO_DIR) / f"experiment_report_{run_name}.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"Wrote {report_path}")

combined_path = Path(REPO_DIR) / "experiment_reports.json"
with open(combined_path, "w") as f:
    json.dump(reports[0] if len(reports) == 1 else reports, f, indent=2)
print(f"Wrote {combined_path}")

display(reports[0] if len(reports) == 1 else reports)


Wrote /content/Binoculars/experiment_report_hc3_unified_t5_perturbed_ai_clean.json
Wrote /content/Binoculars/experiment_reports.json


{'experiment_name': 'Binoculars HC3 evaluation 2',
 'detection_method': 'Binoculars',
 'model_used': 'tiiuae/falcon-7b + tiiuae/falcon-7b-instruct',
 'dataset_used': 'hc3_unified_t5_perturbed_ai_clean',
 'num_samples': 33097,
 'additional_details': {'additional_models_used': ['tiiuae/falcon-7b',
   'tiiuae/falcon-7b-instruct'],
  'notes': 'Zero-shot Binoculars detector using the configured threshold mode.'},
 'metrics_at_1pct_fpr': {'f1': 0.7619206402134044,
  'accuracy': 0.8705622866120797,
  'precision': 0.968767665347654,
  'recall': 0.6278622458325701,
  'auc_roc': 0.9748988399032564},
 'metrics_at_0.1pct_fpr': {'f1': 0.21853118384627912,
  'accuracy': 0.7100039278484455,
  'precision': 0.9838709677419355,
  'recall': 0.12291628503388899,
  'auc_roc': 0.9748988399032564}}